# 01 — Data understanding and exploratory analysis

**Purpose:** validate the source tables and understand the population before building a model. Run from top to bottom.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
ROOT = Path.cwd()
if not (ROOT / 'data' / 'raw').exists() and (ROOT.parent / 'data' / 'raw').exists():
    ROOT = ROOT.parent
DATA = ROOT / 'data' / 'raw'
if not DATA.exists():
    DATA = Path.cwd()

def find_csv(name):
    candidates = [DATA / name, Path.cwd() / name, Path.cwd().parent / 'data' / 'raw' / name]
    for p in candidates:
        if p.exists(): return p
    raise FileNotFoundError(f'Could not find {name}. Place the source CSVs in data/raw/.')

pipeline = pd.read_csv(find_csv('sales_pipeline.csv'))
accounts = pd.read_csv(find_csv('accounts.csv'))
products = pd.read_csv(find_csv('products.csv'))
teams = pd.read_csv(find_csv('sales_teams.csv'))
print(f'Pipeline: {pipeline.shape}; accounts: {accounts.shape}; products: {products.shape}; teams: {teams.shape}')
pipeline.head()

## Data-quality checks

Check the actual columns, types, duplicates, missingness and stage distribution before making assumptions.

In [ ]:
display(pipeline.dtypes.to_frame('dtype'))
display(pipeline.isna().mean().sort_values(ascending=False).rename('missing_fraction').to_frame())
print('Duplicate opportunity IDs:', pipeline['opportunity_id'].duplicated().sum() if 'opportunity_id' in pipeline else 'ID column not found')
if 'deal_stage' in pipeline:
    display(pipeline['deal_stage'].value_counts(dropna=False).rename_axis('deal_stage').to_frame('opportunities'))
    ax = sns.countplot(data=pipeline, x='deal_stage', order=pipeline['deal_stage'].value_counts().index)
    ax.set(title='Opportunity stage distribution', xlabel='Deal stage', ylabel='Count')
    plt.xticks(rotation=20); plt.tight_layout(); plt.show()

## Won/Lost target distribution

Open opportunities are shown for context but are not treated as losses in the modeling phase.

In [ ]:
decided = pipeline[pipeline['deal_stage'].isin(['Won', 'Lost'])].copy()
if not decided.empty:
    display(decided['deal_stage'].value_counts(normalize=True).mul(100).round(2).rename('percent').to_frame())
    ax = sns.countplot(data=decided, x='deal_stage', order=['Won','Lost'])
    ax.set(title='Completed opportunities only', xlabel='Outcome', ylabel='Count')
    plt.tight_layout(); plt.show()
else:
    print('No Won/Lost records found; verify the deal_stage labels.')

## Interpretation checklist

- Confirm the target labels and source row counts.
- Review missing account and product values before joining.
- Keep raw CSVs unchanged; normalize known inconsistencies only in derived columns.
- Do not infer causal drivers from descriptive win-rate differences.